# Instacart — Data Cleaning and Transformation

This notebook transforms the imported data into cleaned, standardized, and enriched tables stored in the `cleaned_data` layer.

The main steps include:

- handling the product with missing category information;
- keeping legitimate null values;
- removing technical ingestion columns that are no longer needed;
- standardizing names and formats;
- combining the `prior` and `train` order-product tables;
- creating useful business variables;
- saving the final datasets as Delta tables in `cleaned_data`.


## 1. Create the `cleaned_data` Schema

This step creates the `cleaned_data` schema where the cleaned and transformed tables will be stored.

The schema is created only if it does not already exist.

In [0]:
from pyspark.sql import functions as F

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.cleaned_data
""")

print("cleaned_data schema created successfully.")

Schéma cleaned_data créé avec succès.


## 2. Load Imported Tables

This step loads the six tables from the `imported_data` layer so they can be cleaned and transformed.

The tables used are:

- `departments`
- `aisles`
- `products`
- `orders`
- `order_products_prior`
- `order_products_train`


In [0]:
departments_imported_data_df = spark.table(
    "workspace.imported_data.departments"
)

aisles_imported_data_df = spark.table(
    "workspace.imported_data.aisles"
)

products_imported_data_df = spark.table(
    "workspace.imported_data.products"
)

orders_imported_data_df = spark.table(
    "workspace.imported_data.orders"
)

prior_imported_data_df = spark.table(
    "workspace.imported_data.order_products_prior"
)

train_imported_data_df = spark.table(
    "workspace.imported_data.order_products_train"
)

print("Toutes les tables importées ont été chargées.")

Toutes les tables importées ont été chargées.


## 3. Identify the Missing Product Categories

The previous data-quality checks showed that one product has missing `aisle_id` and `department_id` values.

Before fixing this product, this step finds the IDs assigned to the `"missing"` aisle and department categories.

These IDs will later be used to replace the missing references in the `products` table.

In [0]:
from pyspark.sql import functions as F

missing_department_id = (
    departments_imported_data_df
    .filter(
        F.lower(F.trim(F.col("department"))) == "missing"
    )
    .select("department_id")
    .first()["department_id"]
)

missing_aisle_id = (
    aisles_imported_data_df
    .filter(
        F.lower(F.trim(F.col("aisle"))) == "missing"
    )
    .select("aisle_id")
    .first()["aisle_id"]
)

print("Missing department_id :", missing_department_id)
print("Missing aisle_id :", missing_aisle_id)

Missing department_id : 21
Missing aisle_id : 100


## 4. Clean and Standardize Departments and Aisles

This step cleans and standardizes the `departments` and `aisles` tables.

The transformations include:

- converting ID columns to integer type
- removing extra spaces from text values
- converting names to lowercase
- keeping the source file information
- keeping the original ingestion timestamp
- adding a new timestamp for the cleaning step

This makes the reference tables consistent and ready to be saved in the `cleaned_data` layer.

In [0]:
# Clean and standardize departments and aisles
departments_cleaned_data_df = (
    departments_imported_data_df
    .select(
        F.col("department_id").cast("int"),  
        F.lower(
            F.trim(F.col("department"))
        ).alias("department"),
        F.col("_source_file"),
        F.col("_ingested_at").alias("_imported_data_ingested_at")
    )
    .withColumn(
        "_cleaned_data_processed_at",
        F.current_timestamp()
    )
)

aisles_cleaned_data_df = (
    aisles_imported_data_df
    .select(
        F.col("aisle_id").cast("int"),
        F.lower(
            F.trim(F.col("aisle"))
        ).alias("aisle"),
        F.col("_source_file"),
        F.col("_ingested_at").alias("_imported_data_ingested_at")  
    )
    .withColumn(
        "_cleaned_data_processed_at",
        F.current_timestamp()
    )
)

display(departments_cleaned_data_df)
display(aisles_cleaned_data_df)

department_id,department,_source_file,_imported_data_ingested_at,_cleaned_data_processed_at
1,frozen,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
2,other,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
3,bakery,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
4,produce,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
5,alcohol,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
6,international,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
7,beverages,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
8,pets,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
9,dry goods pasta,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z
10,bulk,departments.csv,2026-07-26T23:04:52.447Z,2026-07-27T00:04:54.900Z


aisle_id,aisle,_source_file,_imported_data_ingested_at,_cleaned_data_processed_at
1,prepared soups salads,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
2,specialty cheeses,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
3,energy granola bars,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
4,instant foods,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
5,marinades meat preparation,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
6,other,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
7,packaged meat,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
8,bakery desserts,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
9,pasta sauce,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z
10,kitchen supplies,aisles.csv,2026-07-26T23:04:45.048Z,2026-07-27T00:04:55.528Z


## 5. Clean Product Names

This function cleans product names before saving them in the `cleaned_data` layer.

It performs the following steps:

- removes spaces at the beginning and end
- replaces double quotation marks with a single quotation mark
- removes quotation marks that appear at the beginning or end of a product name
- replaces multiple spaces with a single space

This helps keep product names clean and consistent.

In [0]:
from pyspark.sql import functions as F


def nettoyer_nom_produit(colonne):
  

    resultat = F.trim(colonne)

    # Example : 5"" Scissors devient 5" Scissors
    resultat = F.regexp_replace(
        resultat,
        r'""',
        '"'
    )

    # Remove an extra quotation mark at the beginning
    resultat = F.regexp_replace(
        resultat,
        r'^"',
        ''
    )

    # Remove an extra quotation mark at the end
    resultat = F.regexp_replace(
        resultat,
        r'"$',
        ''
    )

    # Replace multiple spaces with one space
    resultat = F.regexp_replace(
        resultat,
        r'\s+',
        ' '
    )

    return resultat

## 6. Clean and Standardize the Products Table

This step cleans and prepares the `products` table for the `cleaned_data` layer.

The transformations include:

- cleaning product names
- checking whether a product name was changed during cleaning
- identifying products with missing category information
- replacing missing `aisle_id` and `department_id` values with the `"missing"` category IDs
- converting ID columns to integer type
- keeping useful metadata for tracking the data
- adding a timestamp for the cleaning step


In [0]:
products_cleaned_data_df = (
    products_imported_data_df

     # Create a cleaned version of the product name
    .withColumn(
        "product_name_clean",
        nettoyer_nom_produit(
            F.col("product_name")
        )
    )

    # Check whether the product name was changed
    .withColumn(
        "product_name_was_cleaned",
        F.col("product_name")
        != F.col("product_name_clean")
    )

    # Check whether category information was missing
    .withColumn(
        "category_was_missing",
        F.col("aisle_id").isNull()
        | F.col("department_id").isNull()
    )

    # Replace missing aisle_id with the "missing" aisle ID
    .withColumn(
        "aisle_id",
        F.coalesce(
            F.col("aisle_id"),
            F.lit(missing_aisle_id)
        ).cast("int")
    )

    # Replace missing department_id with the "missing" department ID
    .withColumn(
        "department_id",
        F.coalesce(
            F.col("department_id"),
            F.lit(missing_department_id)
        ).cast("int")
    )

    # Select and organize the final columns
    .select(
        F.col("product_id").cast("int"),
        F.col("product_name_clean").alias("product_name"),
        F.col("aisle_id"),
        F.col("department_id"),
        F.col("product_name_was_cleaned"),
        F.col("category_was_missing"),
        F.col("_source_file"),
        F.col("_ingested_at").alias("_imported_data_ingested_at")
    )

    # Add the cleaning timestamp 
    .withColumn(
        "_cleaned_data_processed_at",
        F.current_timestamp()
    )
)

print("Products table cleaned successfully.")

La transformation cleaned_data de la table products est terminée.


## 7. Validate the Cleaned Products Table

This step checks that the changes made to the `products` table were applied correctly.

The validation checks:

- the product that originally had missing category information
- the number of products whose missing categories were replaced
- the product names that were changed during cleaning

### 7.1 Check the Corrected Product

Product `6816` was the product with missing `aisle_id` and `department_id`.

This check confirms that:

- its product name was cleaned
- the missing `aisle_id` was replaced
- the missing `department_id` was replaced
- the changes are correctly recorded in the tracking columns

In [0]:
display(
    products_cleaned_data_df
    .filter(F.col("product_id") == 6816)
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "department_id",
        "product_name_was_cleaned",
        "category_was_missing"
    )
)

product_id,product_name,aisle_id,department_id,product_name_was_cleaned,category_was_missing
6816,"Scotch Kids 5"" Scissors",100,21,true,true


### 7.2 Check Products with Replaced Categories

This check counts how many products originally had missing category information.

These products were assigned to the `"missing"` aisle and department categories during the cleaning step.

In [0]:
### 7.1 Check the Corrected Product
display(
    products_cleaned_data_df
    .filter(F.col("product_id") == 6816)
)

product_id,product_name,aisle_id,department_id,product_name_was_cleaned,category_was_missing,_source_file,_imported_data_ingested_at,_cleaned_data_processed_at
6816,"Scotch Kids 5"" Scissors",100,21,true,true,products.csv,2026-07-26T23:04:55.994Z,2026-07-27T00:04:58.183Z


In [0]:
print(
    "Nombre de produits dont la catégorie a été imputée :",
    products_cleaned_data_df
    .filter(F.col("category_was_missing"))
    .count()
)

Nombre de produits dont la catégorie a été imputée : 1


### 7.3 Check Cleaned Product Names

This check shows the product names that were changed during the cleaning process.

The `product_name_was_cleaned` column identifies the products whose names required cleaning.

In [0]:
products_names_cleaned_df = (
    products_cleaned_data_df
    .filter(F.col("product_name_was_cleaned"))
    .select(
        "product_id",
        "product_name",
        "product_name_was_cleaned"
    )
)

display(products_names_cleaned_df)

print(
    "Number of cleaned products :",
    products_names_cleaned_df.count()
)

product_id,product_name,product_name_was_cleaned
18,Pizza for One Suprema Frozen Pizza,true
80,French Tarragon Wine Vinegar,true
105,"Easy Grab 9""x13\"" Oblong Glass Bakeware",true
153,"Fabric Refresher Meadows & Rain Air Freshener (1 Count, 27 oz) Air Care",true
218,"6"" Organic Carrot Cake",true
271,TAI PEI 14OZ.SWEET & SOUR CHKN,true
273,Thin Stackers Brown Rice Salt Free,true
483,"""Constant Comment\"" Decaffeinated Black Tea Blend",true
565,Light and Fluffy Blueberry Pancake Mix,true
578,"10"" Bamboo Skewers",true


Nombre de noms de produits nettoyés : 526


### 7.4 Check Remaining Null Values

This check verifies that there are no missing values left in the main columns of the cleaned `products` table.

The following columns are checked:

- `product_id`
- `product_name`
- `aisle_id`
- `department_id`

After the cleaning step, all of these columns should contain valid values.

In [0]:
products_cleaned_data_quality_df = (
    products_cleaned_data_df
    .agg(
        F.sum(
            F.when(
                F.col("product_id").isNull(),
                1
            ).otherwise(0)
        ).alias("product_id_null"),

        F.sum(
            F.when(
                F.col("product_name").isNull(),
                1
            ).otherwise(0)
        ).alias("product_name_null"),

        F.sum(
            F.when(
                F.col("aisle_id").isNull(),
                1
            ).otherwise(0)
        ).alias("aisle_id_null"),

        F.sum(
            F.when(
                F.col("department_id").isNull(),
                1
            ).otherwise(0)
        ).alias("department_id_null")
    )
)

display(products_cleaned_data_quality_df)

product_id_null,product_name_null,aisle_id_null,department_id_null
0,0,0,0


## 8. Save the Cleaned Tables

This step saves the cleaned reference tables in the `cleaned_data` layer.

The following tables are saved:

- `departments`
- `aisles`
- `products`

The tables are stored in Delta format and will be used in the next transformation steps.

In [0]:
(
    departments_cleaned_data_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cleaned_data.departments")
)

(
    aisles_cleaned_data_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cleaned_data.aisles")
)

(
    products_cleaned_data_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cleaned_data.products")
)

print("The 3 first tables to clean are created .")

Les trois premières tables à nettoyer ont été créées.


### 8.1 Validate the Saved Products Table

This check verifies that the cleaned `products` table was saved correctly in the `cleaned_data` layer.

Product `6816` is checked again to confirm that:

- its product name is cleaned
- its missing `aisle_id` was replaced
- its missing `department_id` was replaced
- the cleaning flags were saved correctly

This confirms that the changes made earlier are present in the final Delta table.

In [0]:

display(
    spark.table("workspace.cleaned_data.products")
    .filter(F.col("product_id") == 6816)
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "department_id",
        "product_name_was_cleaned",
        "category_was_missing"
    )
)

product_id,product_name,aisle_id,department_id,product_name_was_cleaned,category_was_missing
6816,"Scotch Kids 5"" Scissors",100,21,true,true


### 8.2 Check the Saved Tables

This check confirms which tables are currently stored in the `cleaned_data` layer.

At this stage, the following cleaned tables should be available:

- `departments`
- `aisles`
- `products`

The `orders` table may also appear if it was created earlier in the notebook.

In [0]:
#verif
display(
    spark.sql("SHOW TABLES IN workspace.cleaned_data")
)

database,tableName,isTemporary
cleaned_data,aisles,false
cleaned_data,departments,false
cleaned_data,orders,false
cleaned_data,products,false


### 8.3 Summary of the Cleaned Reference Tables

The reference tables have now been cleaned and saved in the `cleaned_data` layer.

**Departments and aisles**

The cleaning steps included:

- converting ID columns to integer type
- removing extra spaces
- converting names to lowercase
- keeping useful tracking columns

**Products**

The cleaning steps included:

- cleaning product names
- correcting product `6816`
- replacing missing category values with `aisle_id = 100` and `department_id = 21`
- adding tracking columns
- checking that no unexpected null values remain

## 9. Clean and Transform the Orders Table

This step cleans the `orders` table and creates new columns that will be useful for the analysis.

The transformations include:

- cleaning the `eval_set` values
- identifying each customer's first order
- identifying whether an order has a previous order
- identifying the customer's final order
- grouping order hours into periods of the day
- converting columns to the correct data types
- keeping the valid null values in `days_since_prior_order`
- keeping useful tracking information
- adding the cleaning timestamp

In [0]:
from pyspark.sql import functions as F


orders_cleaned_data_df = (
    orders_imported_data_df

    # Clean the eval_set values
    # Example: " Train " becomes "train"
    .withColumn(
        "eval_set",
        F.lower(F.trim(F.col("eval_set")))
    )

    # Identify the customer's first order
    .withColumn(
        "is_first_order",
        F.col("order_number") == 1
    )

    # Check whether the customer has a previous order
    .withColumn(
        "has_prior_order",
        F.col("order_number") > 1
    )

    # Identify the customer's final order
    .withColumn(
        "is_final_order",
        F.col("eval_set").isin("train", "test")
    )

    # Group order hours into periods of the day
    .withColumn(
        "order_time_period",
        F.when(
            F.col("order_hour_of_day").between(5, 11),
            "morning"
        )
        .when(
            F.col("order_hour_of_day").between(12, 16),
            "afternoon"
        )
        .when(
            F.col("order_hour_of_day").between(17, 21),
            "evening"
        )
        .otherwise("night")
    )

    # Select and organize the final columns
    .select(
        F.col("order_id").cast("int"),
        F.col("user_id").cast("int"),
        F.col("eval_set"),
        F.col("order_number").cast("int"),
        F.col("order_dow").cast("int"),
        F.col("order_hour_of_day").cast("int"),
        F.col("order_time_period"),

        # Keep these null values because they represent first orders
        F.col("days_since_prior_order").cast("double"),

        F.col("is_first_order"),
        F.col("has_prior_order"),
        F.col("is_final_order"),

         # Tracking information
        F.col("_source_file"),
        F.col("_ingested_at").alias("_imported_data_ingested_at")
    )

    # Add the cleaning timestamp
    .withColumn(
        "_cleaned_data_processed_at",
        F.current_timestamp()
    )
)

print("Orders table cleaned and transformed successfully.")

Transformation de la table orders terminée.


### 9.1 Validate the Cleaned Orders Table

This check verifies that the cleaned `orders` table follows the expected rules.

The validation checks:

- the total number of orders
- whether any `order_id` values are missing
- whether any `user_id` values are missing
- whether first orders incorrectly have a value in `days_since_prior_order`
- whether later orders are missing `days_since_prior_order`
- whether `eval_set` contains only `prior`, `train`, or `test`
- whether `order_hour_of_day` is between 0 and 23

All error counts should be equal to 0.

In [0]:

orders_cleaned_data_quality_df = (
    orders_cleaned_data_df
    .agg(
        # Total number of orders
        F.count("*").alias("nombre_lignes"),

         # order_id should never be null
        F.sum(
            F.when(
                F.col("order_id").isNull(),
                1
            ).otherwise(0)
        ).alias("order_id_null"),

        # user_id should never be null
        F.sum(
            F.when(
                F.col("user_id").isNull(),
                1
            ).otherwise(0)
        ).alias("user_id_null"),

         # First orders should not have days_since_prior_order
        F.sum(
            F.when(
                F.col("is_first_order")
                & F.col("days_since_prior_order").isNotNull(),
                1
            ).otherwise(0)
        ).alias("premieres_commandes_avec_delai"),

         # Later orders should have days_since_prior_order
        F.sum(
            F.when(
                F.col("has_prior_order")
                & F.col("days_since_prior_order").isNull(),
                1
            ).otherwise(0)
        ).alias("commandes_suivantes_sans_delai"),

        # eval_set should contain only prior, train, or test
        F.sum(
            F.when(
                ~F.col("eval_set").isin("prior", "train", "test"),
                1
            ).otherwise(0)
        ).alias("eval_set_invalide"),

        # Order hour should be between 0 and 23
        F.sum(
            F.when(
                ~F.col("order_hour_of_day").between(0, 23),
                1
            ).otherwise(0)
        ).alias("heure_invalide")
    )
)

display(orders_cleaned_data_quality_df)

nombre_lignes,order_id_null,user_id_null,premieres_commandes_avec_delai,commandes_suivantes_sans_delai,eval_set_invalide,heure_invalide
3421083,0,0,0,0,0,0


### 9.2 Save the Cleaned Orders Table

This step saves the cleaned `orders` DataFrame as a Delta table in the `cleaned_data` layer.

The table is saved as:

`workspace.cleaned_data.orders`

In [0]:
(
    orders_cleaned_data_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cleaned_data.orders")
)

print("Cleaned orders table saved successfully.")

La table workspace.cleaned_data.orders a été créée avec succès.


### 9.3 Check the Saved Orders Table

This check confirms that the cleaned `orders` table was saved correctly.

A sample of the saved columns is displayed to verify that the new fields created during the transformation are present.

In [0]:


display(
    spark.table("workspace.cleaned_data.orders")
    .select(
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "days_since_prior_order",
        "is_first_order",
        "has_prior_order",
        "is_final_order",
        "order_time_period"
    )
    .orderBy("user_id", "order_number")
    .limit(20)
)

order_id,user_id,eval_set,order_number,days_since_prior_order,is_first_order,has_prior_order,is_final_order,order_time_period
2539329,1,prior,1,null,true,false,false,morning
2398795,1,prior,2,15.0,false,true,false,morning
473747,1,prior,3,21.0,false,true,false,afternoon
2254736,1,prior,4,29.0,false,true,false,morning
431534,1,prior,5,28.0,false,true,false,afternoon
3367565,1,prior,6,19.0,false,true,false,morning
550135,1,prior,7,20.0,false,true,false,morning
3108588,1,prior,8,14.0,false,true,false,afternoon
2295261,1,prior,9,0.0,false,true,false,afternoon
2550362,1,prior,10,30.0,false,true,false,morning


## 10. Prepare the Order-Product Tables

This step prepares the `order_products_prior` and `order_products_train` tables before combining them.

The transformations include:

- converting the columns to the correct data types
- adding an `order_set` column to show whether the row comes from `prior` or `train`
- creating the boolean column `is_reordered`
- keeping useful tracking information
- adding the cleaning timestamp
- making sure both DataFrames have the same structure so they can be combined later

In [0]:

order_products_prior_cleaned_data_df = (
    prior_imported_data_df

    # Standardize data types
    .select(
        F.col("order_id").cast("int"),
        F.col("product_id").cast("int"),
        F.col("add_to_cart_order").cast("int"),
        F.col("reordered").cast("int"),


        F.col("_source_file"),
        F.col("_ingested_at").alias("_imported_data_ingested_at")
    )

     # Identify the source as prior orders
    .withColumn(
        "order_set",
        F.lit("prior")
    )

    # Create an easier-to-read boolean indicator
    .withColumn(
        "is_reordered",
        F.col("reordered") == 1
    )

    # Add the cleaning timestamp 
    .withColumn(
        "_cleaned_data_processed_at",
        F.current_timestamp()
    )
)


# Prepare order_products_train
order_products_train_cleaned_data_df = (
    train_imported_data_df

    # Standardize data types
    .select(
        F.col("order_id").cast("int"),
        F.col("product_id").cast("int"),
        F.col("add_to_cart_order").cast("int"),
        F.col("reordered").cast("int"),

        # Tracking information
        F.col("_source_file"),
        F.col("_ingested_at").alias("_imported_data_ingested_at")
    )

    # Identify the source as train orders
    .withColumn(
        "order_set",
        F.lit("train")
    )

    # Create an easier-to-read boolean indicator
    .withColumn(
        "is_reordered",
        F.col("reordered") == 1
    )

    # Add the cleaning timestamp 
    .withColumn(
        "_cleaned_data_processed_at",
        F.current_timestamp()
    )
)

print("Prior and train order-product tables prepared successfully.")

Préparation des tables prior et train terminée.


### 10.1 Validate the Prepared Order-Product Tables

This check verifies that the prepared `prior` and `train` tables follow the expected rules.

The validation checks:

- the total number of rows
- missing `order_id` values
- missing `product_id` values
- missing `add_to_cart_order` values
- missing or invalid `reordered` values
- whether `is_reordered` correctly matches `reordered`
- whether each row has the correct `order_set`

All error counts should be equal to 0.

In [0]:

def controler_table_commandes_produits(
    dataframe,
    table_name,
    order_set_attendu
):
    """
    Check the quality of a prepared order-product table.

    """

    return (
        dataframe
        .agg(
            F.count("*").alias("nombre_lignes"),

            F.sum(
                F.when(F.col("order_id").isNull(), 1)
                .otherwise(0)
            ).alias("order_id_null"),

            F.sum(
                F.when(F.col("product_id").isNull(), 1)
                .otherwise(0)
            ).alias("product_id_null"),

            F.sum(
                F.when(F.col("add_to_cart_order").isNull(), 1)
                .otherwise(0)
            ).alias("position_panier_null"),

            F.sum(
                F.when(F.col("reordered").isNull(), 1)
                .otherwise(0)
            ).alias("reordered_null"),

            
            F.sum(
                F.when(
                    ~F.col("reordered").isin(0, 1),
                    1
                ).otherwise(0)
            ).alias("reordered_invalide"),

            
            F.sum(
                F.when(
                    F.col("is_reordered")
                    != (F.col("reordered") == 1),
                    1
                ).otherwise(0)
            ).alias("is_reordered_incoherent"),

            
            F.sum(
                F.when(
                    F.col("order_set") != order_set_attendu,
                    1
                ).otherwise(0)
            ).alias("order_set_incorrect")
        )
        .withColumn(
            "table_name",
            F.lit(table_name)
        )
        .select(
            "table_name",
            "nombre_lignes",
            "order_id_null",
            "product_id_null",
            "position_panier_null",
            "reordered_null",
            "reordered_invalide",
            "is_reordered_incoherent",
            "order_set_incorrect"
        )
    )


controle_prior_df = controler_table_commandes_produits(
    order_products_prior_cleaned_data_df,
    "order_products_prior",
    "prior"
)

controle_train_df = controler_table_commandes_produits(
    order_products_train_cleaned_data_df,
    "order_products_train",
    "train"
)

controle_etape_14_df = (
    controle_prior_df
    .unionByName(controle_train_df)
)

display(controle_etape_14_df)

table_name,nombre_lignes,order_id_null,product_id_null,position_panier_null,reordered_null,reordered_invalide,is_reordered_incoherent,order_set_incorrect
order_products_prior,32434489,0,0,0,0,0,0,0
order_products_train,1384617,0,0,0,0,0,0,0


### 10.2 Combine the Prior and Train Order-Product Tables

This step combines `order_products_prior` and `order_products_train` into one cleaned DataFrame.

The two tables already have the same columns, so `unionByName()` can be used to match columns by name.

The final table keeps:

- `order_id`
- `product_id`
- `add_to_cart_order`
- `reordered`
- `is_reordered`
- `order_set`
- source and processing information

In [0]:


order_products_cleaned_data_df = (
    order_products_prior_cleaned_data_df

    .unionByName(
        order_products_train_cleaned_data_df
    )

    .select(
        "order_id",
        "product_id",
        "add_to_cart_order",
        "reordered",
        "is_reordered",
        "order_set",
        "_source_file",
        "_imported_data_ingested_at",
        "_cleaned_data_processed_at"
    )
)

print("Prior and train order-product tables combined successfully.")

Regroupement des tables prior et train terminé.


### 10.3 Validate the Combined Order-Product Table

This check verifies that the combined `order_products` table is correct after merging the `prior` and `train` data.

The validation checks:

- the total number of rows
- missing `order_id` values
- missing `product_id` values
- missing `add_to_cart_order` values
- invalid `reordered` values
- invalid `order_set` values
- whether `is_reordered` correctly matches `reordered`

All error counts should be equal to 0.

In [0]:

controle_etape_15_df = (
    order_products_cleaned_data_df
    .agg(
        F.count("*").alias("nombre_lignes_total"),

        F.sum(
            F.when(F.col("order_id").isNull(), 1)
            .otherwise(0)
        ).alias("order_id_null"),

        F.sum(
            F.when(F.col("product_id").isNull(), 1)
            .otherwise(0)
        ).alias("product_id_null"),

        F.sum(
            F.when(F.col("add_to_cart_order").isNull(), 1)
            .otherwise(0)
        ).alias("position_panier_null"),

        F.sum(
            F.when(
                ~F.col("reordered").isin(0, 1),
                1
            ).otherwise(0)
        ).alias("reordered_invalide"),

        F.sum(
            F.when(
                ~F.col("order_set").isin("prior", "train"),
                1
            ).otherwise(0)
        ).alias("order_set_invalide"),

        F.sum(
            F.when(
                F.col("is_reordered")
                != (F.col("reordered") == 1),
                1
            ).otherwise(0)
        ).alias("is_reordered_incoherent")
    )
)

display(controle_etape_15_df)

nombre_lignes_total,order_id_null,product_id_null,position_panier_null,reordered_invalide,order_set_invalide,is_reordered_incoherent
33819106,0,0,0,0,0,0


### 10.4 Check the Number of Rows by Order Set

This check confirms that the combined table still contains the expected number of rows from each source.

The rows are grouped by `order_set` to separate:

- `prior` orders
- `train` orders

This helps confirm that no rows were lost when the two tables were combined.

In [0]:

display(
    order_products_cleaned_data_df
    .groupBy("order_set")
    .count()
    .orderBy("order_set")
)

order_set,count
prior,32434489
train,1384617


### 10.5 Save the Combined Order-Product Table

This step saves the combined `order_products` DataFrame in the `cleaned_data` layer.

The final table is saved as:

`workspace.cleaned_data.order_products`

It contains both `prior` and `train` order-product records in one Delta table.

In [0]:

(
    order_products_cleaned_data_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.cleaned_data.order_products")
)

print("Combined order-products table saved successfully.")

La table workspace.cleaned_data.order_products a été créée avec succès.


### 10.6 Check the Saved Order-Product Table

This check confirms that the combined `order_products` table was saved correctly.

The number of rows in the saved Delta table should match the number of rows in the combined DataFrame.

In [0]:
#verif
order_products_saved_count = (
    spark.table("workspace.cleaned_data.order_products")
    .count()
)

print(
    "Number of saved rows:",
    order_products_saved_count
)

Nombre de lignes enregistrées : 33819106


## 11. Final Check of the Cleaned Tables

This final check compares the number of rows saved in each `cleaned_data` table with the expected number of rows.

The following tables are checked:

- `departments`
- `aisles`
- `products`
- `orders`
- `order_products`

For each table, the check shows:

- the expected number of rows
- the actual number of saved rows
- the difference between the two values
- the final status

If the difference is 0, the table is marked as `OK`.

In [0]:


from pyspark.sql import functions as F

expected_counts = {
    "departments": 21,
    "aisles": 134,
    "products": 49688,
    "orders": 3421083,
    "order_products": 33819106
}

verification_results = []

for table_name, expected_count in expected_counts.items():

    full_table_name = f"workspace.cleaned_data.{table_name}"

    # Count the rows saved in the cleaned table
    actual_count = spark.table(full_table_name).count()

     # Compare actual and expected row counts
    difference = actual_count - expected_count

    # Final status
    status = "OK" if difference == 0 else "Check required"

    verification_results.append(
        (
            table_name,
            expected_count,
            actual_count,
            difference,
            status
        )
    )

# Create a summary table
cleaned_data_final_verification_df = spark.createDataFrame(
    verification_results,
    [
        "table_name",
        "expected_count",
        "actual_count",
        "difference",
        "status"
    ]
)

display(
    cleaned_data_final_verification_df.orderBy(
        F.desc("actual_count")
    )
)

table_name,expected_count,actual_count,difference,status
order_products,33819106,33819106,0,OK
orders,3421083,3421083,0,OK
products,49688,49688,0,OK
aisles,134,134,0,OK
departments,21,21,0,OK
